# Kaggle FLUX.2 Klein worker — backend bridge

This notebook runs the FLUX.2 Klein worker on a Kaggle T4 and exposes it to the
FastAPI backend (`RemoteFluxKaggleProvider`) through an ngrok tunnel:

    backend -> POST {tunnel}/generate -> worker (T4 + FLUX.2 Klein) -> {"image": <base64 png>}

The worker script is **ingested from a Kaggle dataset** (single source of truth:
`notebooks/kaggle_flux_worker.py` in the repo) — nothing is pasted inline and the
generation logic inside the script is never edited here.

Run order: **Setup → Secrets → Ingest worker → Start worker → Health → Tunnel →
Smoke test.** Copy the printed PUBLIC URL into the backend's `KAGGLE_GATEWAY_URL`.


In [ ]:
# Setup: FLUX.2 Klein deps + worker server deps.
# pillow is pinned <12.0 because an unpinned upgrade breaks Kaggle's
# preinstalled torchvision (see fix history in the quality-test notebook).
!pip install -q --upgrade https://github.com/huggingface/diffusers/archive/refs/heads/main.zip \
    transformers accelerate sentencepiece protobuf peft matplotlib \
    "pillow<12.0,>=8.0"
!pip install -q fastapi "uvicorn[standard]" pyngrok requests


Overwriting /kaggle/working/kaggle_flux_worker.py


In [ ]:
# Load secrets from Kaggle. Add "HF_TOKEN" and "NGROK_AUTH_TOKEN" under
# Add-ons -> Secrets. HF_TOKEN is required for the gated FLUX.2 Klein model.
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
NGROK_AUTH_TOKEN = secrets.get_secret("NGROK_AUTH_TOKEN")

print("Secrets loaded:", os.environ["HF_TOKEN"][:8] + "...", NGROK_AUTH_TOKEN[:8] + "...")


<function get_pipe at 0x7fc73a6feca0>


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [ ]:
# Ingest the worker script from a Kaggle dataset — no pasting.
# Upload notebooks/kaggle_flux_worker.py (from the repo) into any dataset and
# attach it to this notebook; we locate it and copy it into /kaggle/working.
import glob
import shutil

matches = glob.glob("/kaggle/input/**/kaggle_flux_worker.py", recursive=True)
assert matches, (
    "Attach a dataset containing kaggle_flux_worker.py "
    "(repo: notebooks/kaggle_flux_worker.py). Upload it as a dataset and add it to this notebook."
)
src = matches[0]
shutil.copy(src, "/kaggle/working/kaggle_flux_worker.py")
print("Ingested worker from:", src)


INFO:     127.0.0.1:57920 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok', 'device': 'cuda', 'model_loaded': True}


In [ ]:
# Start the worker in the background: it preloads the model, then serves :8000.
# Logs go to /kaggle/working/worker.log.
!nohup python /kaggle/working/kaggle_flux_worker.py > /kaggle/working/worker.log 2>&1 &
print("Worker starting... tail /kaggle/working/worker.log for progress")


     - 13.8 MB 9.3 MB/s 0:00:01m
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 1.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 23.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 48.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.4 MB/s eta 0:00:00

In [ ]:
# Wait for the worker to finish preloading the model (can take a few minutes).
import time
import requests

for i in range(120):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=5)
        print(r.json())
        break
    except Exception:
        if i % 6 == 0:
            print(f"waiting for worker ({i * 5}s)... check worker.log if it stalls")
        time.sleep(5)
else:
    raise RuntimeError("Worker did not become healthy in time — check worker.log")


In [ ]:
# Expose :8000 to the internet. Copy the PUBLIC URL into the backend's
# KAGGLE_GATEWAY_URL env var (keep this cell running while testing).
from pyngrok import ngrok

ngrok.kill()  # clear any stale tunnel from a previous kernel run
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(8000)
print("PUBLIC URL:", tunnel.public_url)
print("Set KAGGLE_GATEWAY_URL=" + tunnel.public_url + " on the backend.")


Token loaded: hf_JMNXj...


In [ ]:
# Local smoke test through the worker API before wiring the backend.
# Set PRODUCT_IMAGE to a file in your attached product-image dataset.
import base64
import requests
from IPython.display import display
from PIL import Image

PRODUCT_IMAGE = "/kaggle/input/my-product/p1.webp"  # TODO: your dataset path

payload = {
    "prompt": "Luxury studio product photo",
    "product_images": [PRODUCT_IMAGE],
    "reference_images": [],
    "width": 1080,
    "height": 1350,
}

r = requests.post("http://127.0.0.1:8000/generate", json=payload, timeout=600)
print("status:", r.status_code)
data = r.json()
print("response keys:", list(data.keys()))

if r.status_code == 200:
    out = "/kaggle/working/smoke_test.png"
    with open(out, "wb") as f:
        f.write(base64.b64decode(data["image"]))
    print("saved", out)
    display(Image.open(out))


[worker] preloading black-forest-labs/FLUX.2-klein-4B ...
Worker started in background.
